# MVP de Engenharia de Dados
## Notebook 4 — Camada Silver

### Objetivo deste notebook

Este notebook é responsável pelo tratamento, padronização e validação dos dados provenientes da camada Bronze.

A tabela de entrada utilizada será:

`workspace.energia_bronze.balanco_energia_subsistema`

Nesta etapa serão avaliados aspectos de qualidade dos dados e realizadas as transformações necessárias para preparar uma versão consistente do conjunto de dados.

Entre os principais pontos avaliados estão:

- tipos de dados;
- valores nulos;
- registros duplicados;
- consistência dos subsistemas;
- período temporal dos dados;
- conversão dos atributos de geração, carga e intercâmbio para tipos numéricos adequados.

As transformações realizadas serão documentadas e validadas antes da persistência dos dados na camada Silver.

### Entrada

Tabela Delta da camada Bronze:

`workspace.energia_bronze.balanco_energia_subsistema`

### Saída esperada

Tabela tratada e validada no schema:

`workspace.energia_silver`

In [0]:
tabela_bronze = "workspace.energia_bronze.balanco_energia_subsistema"

df_silver = spark.table(tabela_bronze)

print("Tabela Bronze carregada com sucesso.")
print("Quantidade de registros:", df_silver.count())

Tabela Bronze carregada com sucesso.
Quantidade de registros: 263040


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

df_nulos = df_silver.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
])

display(df_nulos)

id_subsistema,nom_subsistema,din_instante,val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio
0,0,0,0,0,0,0,0,0


In [0]:
total_registros = df_silver.count()
total_distintos = df_silver.distinct().count()
duplicados = total_registros - total_distintos

print("Total de registros:", total_registros)
print("Registros distintos:", total_distintos)
print("Registros duplicados:", duplicados)

Total de registros: 263040
Registros distintos: 263040
Registros duplicados: 0


In [0]:
from pyspark.sql.functions import col, trim, sum as spark_sum

colunas_metricas = [
    "val_gerhidraulica",
    "val_gertermica",
    "val_gereolica",
    "val_gersolar",
    "val_carga",
    "val_intercambio"
]

df_vazios = df_silver.select([
    spark_sum(
        (trim(col(c)) == "").cast("int")
    ).alias(c)
    for c in colunas_metricas
])

display(df_vazios)

val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio
0,0,0,0,0,0


In [0]:
from pyspark.sql.functions import expr, col, sum as spark_sum

df_invalidos = df_silver.select([
    spark_sum(
        (
            col(c).isNotNull() &
            expr(f"TRY_CAST(`{c}` AS DOUBLE)").isNull()
        ).cast("int")
    ).alias(c)
    for c in colunas_metricas
])

display(df_invalidos)

val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio
0,0,0,0,0,0


In [0]:
from pyspark.sql.functions import col

for coluna in colunas_metricas:
    df_silver = df_silver.withColumn(
        coluna,
        col(coluna).cast("double")
    )

df_silver.printSchema()

root
 |-- id_subsistema: string (nullable = true)
 |-- nom_subsistema: string (nullable = true)
 |-- din_instante: timestamp (nullable = true)
 |-- val_gerhidraulica: double (nullable = true)
 |-- val_gertermica: double (nullable = true)
 |-- val_gereolica: double (nullable = true)
 |-- val_gersolar: double (nullable = true)
 |-- val_carga: double (nullable = true)
 |-- val_intercambio: double (nullable = true)



In [0]:
df_silver \
    .select("id_subsistema", "nom_subsistema") \
    .distinct() \
    .orderBy("id_subsistema") \
    .show(truncate=False)

+-------------+----------------------------+
|id_subsistema|nom_subsistema              |
+-------------+----------------------------+
|N            |NORTE                       |
|NE           |NORDESTE                    |
|S            |SUL                         |
|SE           |SUDESTE/CENTRO-OESTE        |
|SIN          |SISTEMA INTERLIGADO NACIONAL|
+-------------+----------------------------+



In [0]:
from pyspark.sql.functions import min, max

df_silver.select(
    min("din_instante").alias("primeiro_registro"),
    max("din_instante").alias("ultimo_registro")
).show(truncate=False)

+-------------------+-------------------+
|primeiro_registro  |ultimo_registro    |
+-------------------+-------------------+
|2020-01-01 00:00:00|2025-12-31 23:00:00|
+-------------------+-------------------+



In [0]:
from pyspark.sql.functions import count

df_registros_por_hora = (
    df_silver
    .groupBy("din_instante")
    .agg(count("*").alias("qtd_registros"))
)

df_registros_por_hora \
    .groupBy("qtd_registros") \
    .count() \
    .orderBy("qtd_registros") \
    .show()

+-------------+-----+
|qtd_registros|count|
+-------------+-----+
|            5|52608|
+-------------+-----+



In [0]:
from pyspark.sql.functions import countDistinct

df_subsistemas_por_hora = (
    df_silver
    .groupBy("din_instante")
    .agg(
        countDistinct("id_subsistema").alias("qtd_subsistemas")
    )
)

df_subsistemas_por_hora \
    .groupBy("qtd_subsistemas") \
    .count() \
    .orderBy("qtd_subsistemas") \
    .show()

+---------------+-----+
|qtd_subsistemas|count|
+---------------+-----+
|              5|52608|
+---------------+-----+



## Diagnóstico e tratamento da qualidade dos dados

Antes da persistência dos dados na camada Silver, foram realizadas verificações de qualidade sobre os 263.040 registros provenientes da camada Bronze.

### Resultados das verificações

- **Valores nulos:** não foram identificados valores nulos nas 9 colunas.
- **Registros duplicados:** não foram identificadas linhas completamente duplicadas.
- **Strings vazias:** não foram identificados valores vazios nas seis métricas analisadas.
- **Conversibilidade numérica:** todos os valores das métricas puderam ser convertidos para `double`, sem identificação de valores textuais incompatíveis.
- **Subsistemas identificados:** N, NE, S, SE e SIN.
- **Período temporal:** de 01/01/2020 às 00:00 até 31/12/2025 às 23:00.
- **Horários distintos:** 52.608.
- **Regularidade temporal:** todos os 52.608 horários possuem exatamente 5 registros.
- **Consistência dos subsistemas:** todos os horários possuem 5 identificadores de subsistema distintos.

### Transformação de tipos

Na camada Bronze, os atributos de geração, carga e intercâmbio foram mantidos como `string` para permitir a consolidação dos arquivos anuais, que apresentavam diferenças de schema na fonte.

Após a validação de que todos os valores eram compatíveis com representação numérica, os seguintes atributos foram convertidos para `double` na camada Silver:

- `val_gerhidraulica`
- `val_gertermica`
- `val_gereolica`
- `val_gersolar`
- `val_carga`
- `val_intercambio`

A conversão permite que essas medidas sejam utilizadas corretamente em cálculos, agregações e análises nas etapas seguintes do pipeline.

In [0]:
from pyspark.sql.functions import year, month, dayofmonth, hour

df_silver = (
    df_silver
    .withColumn("ano", year("din_instante"))
    .withColumn("mes", month("din_instante"))
    .withColumn("dia", dayofmonth("din_instante"))
    .withColumn("hora", hour("din_instante"))
)

df_silver.printSchema()

root
 |-- id_subsistema: string (nullable = true)
 |-- nom_subsistema: string (nullable = true)
 |-- din_instante: timestamp (nullable = true)
 |-- val_gerhidraulica: double (nullable = true)
 |-- val_gertermica: double (nullable = true)
 |-- val_gereolica: double (nullable = true)
 |-- val_gersolar: double (nullable = true)
 |-- val_carga: double (nullable = true)
 |-- val_intercambio: double (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- hora: integer (nullable = true)



In [0]:
display(
    df_silver.select(
        "din_instante",
        "ano",
        "mes",
        "dia",
        "hora"
    ).orderBy("din_instante").limit(10)
)

din_instante,ano,mes,dia,hora
2020-01-01T00:00:00.000Z,2020,1,1,0
2020-01-01T00:00:00.000Z,2020,1,1,0
2020-01-01T00:00:00.000Z,2020,1,1,0
2020-01-01T00:00:00.000Z,2020,1,1,0
2020-01-01T00:00:00.000Z,2020,1,1,0
2020-01-01T01:00:00.000Z,2020,1,1,1
2020-01-01T01:00:00.000Z,2020,1,1,1
2020-01-01T01:00:00.000Z,2020,1,1,1
2020-01-01T01:00:00.000Z,2020,1,1,1
2020-01-01T01:00:00.000Z,2020,1,1,1


In [0]:
tabela_silver = "workspace.energia_silver.balanco_energia_subsistema"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(tabela_silver)

print("Tabela Silver criada com sucesso:")
print(tabela_silver)

Tabela Silver criada com sucesso:
workspace.energia_silver.balanco_energia_subsistema


In [0]:
df_validacao_silver = spark.table(
    "workspace.energia_silver.balanco_energia_subsistema"
)

print("Quantidade de registros na Silver:", df_validacao_silver.count())
print("Quantidade de colunas na Silver:", len(df_validacao_silver.columns))

df_validacao_silver.printSchema()

Quantidade de registros na Silver: 263040
Quantidade de colunas na Silver: 13
root
 |-- id_subsistema: string (nullable = true)
 |-- nom_subsistema: string (nullable = true)
 |-- din_instante: timestamp (nullable = true)
 |-- val_gerhidraulica: double (nullable = true)
 |-- val_gertermica: double (nullable = true)
 |-- val_gereolica: double (nullable = true)
 |-- val_gersolar: double (nullable = true)
 |-- val_carga: double (nullable = true)
 |-- val_intercambio: double (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- hora: integer (nullable = true)



## Resultado da camada Silver

O processamento da camada Silver foi concluído com sucesso.

A tabela Bronze foi utilizada como fonte para as verificações de qualidade e para as transformações necessárias à preparação dos dados.

### Qualidade dos dados

As verificações realizadas sobre os 263.040 registros identificaram:

- **0 valores nulos**;
- **0 registros duplicados**;
- **0 strings vazias** nas métricas;
- **0 valores incompatíveis com conversão numérica**;
- presença consistente dos identificadores `N`, `NE`, `S`, `SE` e `SIN`;
- período temporal entre **01/01/2020 00:00** e **31/12/2025 23:00**;
- **52.608 horários distintos**;
- exatamente **5 registros e 5 identificadores distintos em cada horário**.

### Transformações realizadas

As seis métricas de geração, carga e intercâmbio foram convertidas de `string` para `double`:

- `val_gerhidraulica`
- `val_gertermica`
- `val_gereolica`
- `val_gersolar`
- `val_carga`
- `val_intercambio`

Também foram derivados de `din_instante` os atributos:

- `ano`
- `mes`
- `dia`
- `hora`

Esses atributos facilitam as agregações e análises temporais nas próximas etapas do pipeline.

### Resultado

A tabela Silver foi persistida em formato Delta no catálogo:

`workspace.energia_silver.balanco_energia_subsistema`

A tabela resultante possui:

- **263.040 registros**
- **13 atributos**

A quantidade de registros permaneceu inalterada em relação à camada Bronze, indicando que nenhum registro precisou ser descartado durante o processo de tratamento.

A camada Silver passa a ser a fonte de dados tratados e validados para a modelagem analítica que será realizada na camada Gold.
